In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Open the customer picker (CustomerBar button, verified in CustomerPicker.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Select Customer')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//input[@placeholder='Search by name or phone...']")))

    # First row is Walk-in, so the 2nd row is the first saved customer
    rows = driver.find_elements(By.XPATH, "//div[@role='button' and contains(@class, 'pos-row')]")
    assert len(rows) >= 2, "No saved customers found (only Walk-in Customer shown)."
    cust_name = rows[1].text.split("\n")[0]
    print("Selected customer name:", cust_name)
    rows[1].click()
    time.sleep(2)

    # Picker closed and customer bar now shows Change + the name
    wait.until(EC.visibility_of_element_located((By.XPATH, "//button[contains(., 'Change')]")))
    assert cust_name in driver.find_element(By.TAG_NAME, "body").text, "Customer name not shown in POS."

    # Associate with the sale WITHOUT creating a sale: add 1 item to the cart only
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    add_buttons = [b for b in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add_buttons, "No addable medicine found in the catalog."
    add_buttons[0].click()
    time.sleep(2)

    body = driver.find_element(By.TAG_NAME, "body").text
    assert ("CRM Discount \u00b7 " + cust_name) in body, "Customer not associated with the current sale/cart."

    print("Current URL:", driver.current_url)
    print("Page text:", body[:200])
    print("PASS: Customer Selection")
except Exception as e:
    print("FAIL: Customer Selection")
    print("Error:", e)
    driver.save_screenshot("16_customer_selection_FAIL.png")

In [ ]:
driver.quit()